# Working with MERFISH data

This tutorial reads MERFISH data into stLearn, uses the blank control barcodes for quality control, integrates three replicates and clusters the cells.

The data are from [Xia et al. (2019), *PNAS* 116(39):19490](https://doi.org/10.1073/pnas.1912459116): MERFISH profiling of 10,050 genes in cultured U-2 OS cells, imaged in three replicates. We use two of the paper's supplementary files:

- `pnas.1912459116.sd12.csv`: the count matrix, genes × cells, for all three replicates
- `pnas.1912459116.sd15.xlsx`: the cell positions in microns, one sheet per replicate

The original PNAS download links no longer work, so we download the same files from the copy kept in the Giotto team's [spatial-datasets](https://github.com/drieslab/spatial-datasets) repository on GitHub.

In [ ]:
import harmonypy as hm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import stlearn as st

# Setting figure parameters
st.settings.set_figure_params(dpi=120)

Read MERFISH data follow the instruction from scanpy

In [ ]:
import urllib.request
from pathlib import Path

base_url = (
    "https://raw.githubusercontent.com/drieslab/spatial-datasets/master/"
    "data/2019_merFISH_PNAS/raw_data/"
)
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

count_file = data_dir / "pnas.1912459116.sd12.csv"
spatial_file = data_dir / "pnas.1912459116.sd15.xlsx"

for path in (count_file, spatial_file):
    if not path.exists():
        urllib.request.urlretrieve(base_url + path.name, path)

## Read the counts and positions

`st.io.read_merfish` reads this format, but only the first sheet of positions (replicate 1), and it keeps the blank control barcodes as if they were genes. To use all three replicates and handle the blanks, we read the files with pandas and build the stLearn objects with `st.io.create_stlearn`.

In [ ]:
counts = pd.read_csv(count_file, index_col=0)  # genes x cells
positions = pd.read_excel(spatial_file, sheet_name=None, index_col=0)

print(f"Count matrix: {counts.shape[0]:,} barcodes x {counts.shape[1]:,} cells")
for name, pos in positions.items():
    print(f"{name}: {len(pos)} cells")

## Blank controls

MERFISH gives each gene a binary barcode. Of the 12,903 possible barcodes, 2,853 were not assigned to any gene. Any counts on these blank barcodes are decoding errors, so they measure how often a barcode is misread. The paper reports a misidentification rate of 3.9%.

In [ ]:
is_blank = counts.index.str.startswith("Blank-")
blank_counts = counts[is_blank]
gene_counts = counts[~is_blank]

misidentification = blank_counts.mean(axis=1).mean() / gene_counts.mean(axis=1).mean()
print(f"{is_blank.sum():,} blank barcodes, {(~is_blank).sum():,} genes")
print(f"Misidentification rate: {misidentification:.1%}")

Following the paper, we keep the genes whose average count per cell is at least the 95th percentile of the blank barcodes' averages: a gene detected less often than that can't be told apart from decoding errors. The blanks themselves are removed from the matrix.

In [ ]:
threshold = np.percentile(blank_counts.mean(axis=1), 95)
expressed = gene_counts.mean(axis=1) >= threshold
gene_counts = gene_counts[expressed]
print(f"Threshold: {threshold:.2f} counts per cell")
print(f"Kept {expressed.sum():,} of {len(expressed):,} genes")

## One stLearn object per replicate

Each replicate gets its own library, with the positions moved so they start near zero. The positions are in microns, so we also record that one full-resolution pixel is one micron: `st.concatenate_spatial_adata` uses this to draw the replicates at the same scale.

In [ ]:
replicates = []
for name, pos in positions.items():
    library_id = name.replace(" ", "_")
    cells = pos.index
    spatial = (pos - pos.min() + 100).rename(
        columns={"x_microns": "imagecol", "y_microns": "imagerow"}
    )
    adata = st.io.create_stlearn(
        count=gene_counts[cells].T.astype(np.float32),
        spatial=spatial,
        library_id=library_id,
    )
    adata.uns["spatial"][library_id]["scalefactors"]["microns_per_pixel"] = 1.0
    adata.obs["blank_fraction"] = (
        blank_counts[cells].sum() / counts[cells].sum()
    ).to_numpy()
    replicates.append(adata)

data = st.concatenate_spatial_adata(replicates, ncols=3, fixed_size=(1000, 1000))
data.obs["batch"] = data.obs["batch"].cat.rename_categories(list(positions))
data

## Quality check

Each dot is a cell, sized by its total counts and coloured by the number of genes detected.

In [ ]:
plt.rcParams["figure.figsize"] = (12, 4)
st.pl.qc_plot(data, name="merfish_qc")

In [ ]:
data.obs.groupby("batch")["blank_fraction"].describe()[["mean", "min", "max"]]

## Preprocessing

Normalise and log-transform, then keep the 1,600 most variable genes, similar to the 1,598 the paper used. With `batch_key`, the genes are chosen within each replicate, so differences between replicates don't drive the choice.

In [ ]:
st.pp.normalize_total(data)
st.pp.log1p(data)
st.pp.highly_variable_genes(data, n_top_genes=1600, batch_key="batch", subset=True)
st.pp.scale(data)

## PCA and integration with Harmony

Run PCA with 30 components, as in the paper, then correct them for differences between replicates with [Harmony](https://github.com/slowkow/harmonypy), as in the *Integrating Multiple Datasets* tutorial.

In [ ]:
st.em.run_pca(data, n_comps=30, random_state=0)

ho = hm.run_harmony(
    np.ascontiguousarray(data.obsm["X_pca"]), data.obs, "batch", verbose=False
)
data.obsm["X_pca"] = ho.Z_corr

## Leiden clustering

In [ ]:
st.pp.neighbors(data,n_neighbors=25)

In [ ]:
st.tl.clustering.leiden(data, resolution=0.5)

In [ ]:
pd.crosstab(data.obs["leiden"], data.obs["batch"])

In [ ]:
st.pl.cluster_plot(data, use_label="leiden", size=5)